# Tutorial 14_A: Building a Simple RNN for Named Entity Recognition (NER)

This notebook demonstrates how to build and implement a Simple Recurrent Neural Network (RNN) using TensorFlow and Keras to perform Named Entity Recognition. The goal is to classify each word in a sentence into specific categories like PERSON, LOCATION, or ORGANIZATION.

---

## 1. Objectives

* Understand the architecture of a Simple RNN.
* Implement an RNN for sequence labeling.
* Prepare text data and labels for deep learning.
* Visualize and test the results on new sentences.

---

## 2. Prepare the Dataset

We begin by creating a small dataset of sentences and their corresponding NER labels. We will then tokenize the text and convert labels into integers.

In [1]:
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout, TimeDistributed
from sklearn.preprocessing import LabelEncoder

# Sample data
sentences = [
    "Barack Obama was born in Hawaii",
    "Google is based in Mountain View"
]

# Defining labels for each word in the sentences
labels = [
    ["PERSON", "PERSON", "O", "O", "O", "LOCATION", "O"], # Labels for the first sentence
    ["ORGANIZATION", "O", "O", "O", "LOCATION", "O"]      # Labels for the second sentence
]

# Tokenizing the sentences (converting words into integers)
tokenizer = Tokenizer(lower=True)
tokenizer.fit_on_texts(sentences)
X = tokenizer.texts_to_sequences(sentences)

# Padding the sequences to have the same length
X = pad_sequences(X, padding='post')

# Encode the labels
label_encoder = LabelEncoder()
label_encoder.fit(["O", "PERSON", "LOCATION", "ORGANIZATION"])

# Convert labels to numerical values
y = [label_encoder.transform(label) for label in labels]

# Pad the labels so that they match the shape of the input sequences (X)
y = pad_sequences(y, padding="post", maxlen=X.shape[1])

# Reshape y to match the model's output (TimeDistributed layer expects 3D input)
y = np.expand_dims(y, -1)

---

## 3. Discussion: Data Preprocessing

In NER tasks, data preparation is unique because it is a **sequence-to-sequence** problem. Every input token (word) must have a corresponding output token (label).

* **Padding:** We use `padding='post'` to ensure all sentences have the same length by adding zeros at the end.
* **Label Encoding:** Since machines don't understand "PERSON" or "LOCATION", we map them to integers.
* **Dimensionality:** We expand the dimensions of `y` because the model predicts a label for *each* time step in the sequence.

---

## 4. Build the RNN Model

The RNN processes each word sequentially and maintains a hidden state to capture context.

In [2]:
# Model definition
model = Sequential()

# Embedding Layer: Converts word integers into dense vectors
model.add(Embedding(input_dim=len(tokenizer.word_index) + 1, output_dim=50, input_length=X.shape[1]))

# Simple RNN layer: return_sequences=True is essential for many-to-many tasks
model.add(SimpleRNN(units=50, return_sequences=True))

# Dropout to avoid overfitting
model.add(Dropout(0.1))

# TimeDistributed Dense Layer for making predictions at each time step
model.add(TimeDistributed(Dense(len(label_encoder.classes_), activation="softmax")))

# Compile the model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

---

## 5. Train the Model

We fit the model using the processed sequences and labels.

In [3]:
# Train the model
model.fit(np.array(X), np.array(y), epochs=3, batch_size=2)

Epoch 1/3
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - accuracy: 0.1667 - loss: 1.3837
Epoch 2/3
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.2500 - loss: 1.3626
Epoch 3/3
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.4167 - loss: 1.3336


---

## 6. Test the Model with a New Sentence

Once trained, we can pass a completely new sentence through the pipeline to see how the RNN labels each word.

In [4]:
# Test with a new sentence
test_sentence = ["Barack Obama went to Hawaii"]
test_sequence = tokenizer.texts_to_sequences(test_sentence)
test_sequence = pad_sequences(test_sequence, padding='post', maxlen=X.shape[1])

# Predicting the NER labels
predictions = model.predict(test_sequence)

# Decode predictions
decoded_predictions = label_encoder.inverse_transform(np.argmax(predictions, axis=-1)[0])

# Display results
print("\nNER Results:")
for word, label in zip(test_sentence[0].split(), decoded_predictions):
    print(f"Word: {word} | Predicted Label: {label}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 239ms/step

NER Results:
Word: Barack | Predicted Label: PERSON
Word: Obama | Predicted Label: O
Word: went | Predicted Label: ORGANIZATION
Word: to | Predicted Label: O
Word: Hawaii | Predicted Label: O


---

## 7. Discussion: Model Architecture

* **Embedding Layer:** This turns high-dimensional one-hot encoded words into lower-dimensional vectors that capture semantic meaning.
* **SimpleRNN:** The core engine that processes the sequence. By setting `return_sequences=True`, we ensure the model outputs a prediction for every single word, not just the final word of the sentence.
* **TimeDistributed:** This wrapper applies the `Dense` layer (the classifier) to every hidden state produced by the RNN for each time step.

---



# **Task 1: Expanded Custom Dataset**
To improve the model's ability to generalize, we will add more sentences covering a wider range of entities.

In [5]:
# Expanded sample data
custom_sentences = [
    "Barack Obama was born in Hawaii",
    "Google is based in Mountain View",
    "Elon Musk works at Tesla in California",
    "Microsoft was founded by Bill Gates in Washington",
    "Apple is located in Cupertino"
]

# Labels for the expanded dataset
custom_labels = [
    ["PERSON", "PERSON", "O", "O", "O", "LOCATION", "O"],
    ["ORGANIZATION", "O", "O", "O", "LOCATION", "O"],
    ["PERSON", "PERSON", "O", "O", "ORGANIZATION", "O", "LOCATION"],
    ["ORGANIZATION", "O", "O", "O", "PERSON", "PERSON", "O", "LOCATION"],
    ["ORGANIZATION", "O", "O", "O", "LOCATION"]
]

# Re-tokenizing and padding with the new data
tokenizer = Tokenizer(lower=True)
tokenizer.fit_on_texts(custom_sentences)
X_custom = tokenizer.texts_to_sequences(custom_sentences)
X_custom = pad_sequences(X_custom, padding='post')

# Re-encoding labels
y_custom = [label_encoder.transform(label) for label in custom_labels]
y_custom = pad_sequences(y_custom, padding="post", maxlen=X_custom.shape[1])
y_custom = np.expand_dims(y_custom, -1)

# **Task 2: Tuning Hyperparameters**
We will now modify the number of units in the RNN layer, increase the training epochs, and adjust the learning rate to see how it impacts performance.

In [6]:
from tensorflow.keras.optimizers import Adam

# Model with adjusted hyperparameters
model_tuned = Sequential()

# Embedding Layer
model_tuned.add(Embedding(input_dim=len(tokenizer.word_index) + 1,
                          output_dim=64,
                          input_length=X_custom.shape[1]))

# Simple RNN layer with 100 units (Increased from 50)
model_tuned.add(SimpleRNN(units=100, return_sequences=True))

model_tuned.add(Dropout(0.2)) # Increased dropout for regularization

# TimeDistributed Dense Layer
model_tuned.add(TimeDistributed(Dense(len(label_encoder.classes_), activation="softmax")))

# Compiling with a custom learning rate
custom_optimizer = Adam(learning_rate=0.005) # Specific learning rate adjustment
model_tuned.compile(optimizer=custom_optimizer,
                    loss='sparse_categorical_crossentropy',
                    metrics=['accuracy'])

# Training for 20 epochs (Increased from 3)
model_tuned.fit(np.array(X_custom), np.array(y_custom), epochs=20, batch_size=2)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 18ms/step - accuracy: 0.4500 - loss: 1.3273
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6500 - loss: 0.9573
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.7000 - loss: 0.7165
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.8250 - loss: 0.5219
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.8000 - loss: 0.4449
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.8500 - loss: 0.3398
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9250 - loss: 0.2584
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9500 - loss: 0.2224
Epoch 9/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.9750 - loss: 0.1830
Epoch 10/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 1.0000 - loss: 0.1455
Epoch 11/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step - accuracy: 1.0000 - loss: 0.1188
Epoch 12/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 1.0000 - loss: 0.0815
Epoch 13/20


# Discussion: Observations on Tuning

Units: Increasing the RNN units allows the model to learn more complex patterns in the sequences, though it requires more data to avoid overfitting.


Epochs: By increasing the epochs, the model has more opportunities to minimize the loss function, which is particularly useful as the dataset grows.


Learning Rate: A higher learning rate (0.005) speeds up the convergence, but if it is too high, the model might overshoot the optimal weights
# Final Test
Let’s verify the tuned model with a fresh sentence.

In [7]:
test_sentence_final = ["Bill Gates visited Tesla in California"]
test_seq_final = tokenizer.texts_to_sequences(test_sentence_final)
test_seq_final = pad_sequences(test_seq_final, padding='post', maxlen=X_custom.shape[1])

# Predicting
final_predictions = model_tuned.predict(test_seq_final)
decoded_final = label_encoder.inverse_transform(np.argmax(final_predictions, axis=-1)[0])

# Displaying final results
print("\nFinal Tuned NER Results:")
for word, label in zip(test_sentence_final[0].split(), decoded_final):
    print(f"Word: {word} | Predicted Label: {label}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 381ms/step

Final Tuned NER Results:
Word: Bill | Predicted Label: PERSON
Word: Gates | Predicted Label: PERSON
Word: visited | Predicted Label: O
Word: Tesla | Predicted Label: O
Word: in | Predicted Label: LOCATION
Word: California | Predicted Label: LOCATION
